In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="IP2312U_VSET", library="Power_Management_ICs")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
    ## Save part model
    # save_part_model(
    # name="FDMC510P-MS",
    # library="Device",
    # model_content=".MODEL D1 D\n",
    # vendor_provided=False,
    # )


* Injoinic IP2312U_VSET 3A single-cell Li-ion charger
* Behavioral average model for ngspice.
* Captures CC/CV charging profile, VIN UVLO/OVP, NTC derating,
* and LED status drivers. Switching details at SW are not modeled;
* current is delivered directly into BAT for system-level simulation.
*
* Pin order:
*   1 D1   - LED1 / VSET multifunction
*   2 TEST - Test pin (sense to BAT via external resistor)
*   3 D2   - LED2 status output
*   4 NTC  - NTC temperature sense
*   5 BAT  - Battery positive
*   6 ICHG - Charge current set resistor
*   7 SW   - Switch node (not explicitly modeled)
*   8 VIN  - 5 V input
*   9 EP   - Exposed pad, connect to GND
*
.SUBCKT IP2312U_VSET D1 TEST D2 NTC BAT ICHG SW VIN EP

* -------------------------------------------------------------------
* Parameters (can be overridden from the netlist if needed)
* -------------------------------------------------------------------
.PARAM VTRGT     = 4.2     ; Charge target voltage (IP2312 default)
.PARAM VRCH    

In [ ]:
from pathlib import Path
from python.spice_tools import convert_skidl_module

name = convert_skidl_module(
    input_path=Path("test_cases/test_charger_3A/skidl/modules/ip2312_charger.py"),
    subckt_name="IP2312_CHARGER",
    output_path=Path("test_cases/test_charger_3A/spice/ip2312_charger/dut.py"),
)

In [4]:
name

'IP2312_CHARGER_pyspice'

In [ ]:
import json

test_bench_path = "test_cases/case_3A_charger/testbench/battery_protection_schema_valid.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases]

case_ids

In [ ]:
from python.spice_tools.harness_sanity import harness_sanity_check

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    result = harness_sanity_check(harness_path=harness_path)
    print(result)


In [ ]:
from python.spice_tools.testbench_runner import run_use_case

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    reports = run_use_case(
        schema_path=test_bench_path,
        harness_path=harness_path,
        use_case_name=case_ids[idx],
        dut_path="test_cases/case_3A_charger/spice/battery_protection/battery_protection_pyspice.py",
        dut_module_name="Battery_Protection_pyspice",
    )

    print(reports)
